# 導言

本文件說明 Streamlit 績效比對儀表板與 回測結果資料庫 中
所有績效指標的**定義、計算口徑與判讀基準**，並記載機器學習／深度學習策略所使用的
**特徵與訓練資料**及完整執行步驟。所有文獻依據以 APA 第 7 版格式列於文末。

**回測共同設定**：S&P 500 歷史成分股（Tiingo，2000–2025，動態成分股名冊避免存活者偏差）；
形成期 252 交易日、交易期 126 交易日、滾動步長 21 日；初始資本 \$10,000；
單邊交易成本 **0.29%**（進場、出場各按部位名目額扣一次，故每配對一完整往返 ≈ 0.58%），
依 Do 與 Faff（2012）對美股配對交易單邊成本約 30 bps 之估計。


# 報酬與資本口徑

配對交易為市場中性策略，資金**部分時間閒置**（無配對達到進場門檻時持有現金）。
因此「報酬」依分母（資本基準）不同而有多種口徑，各口徑對應不同文獻慣例。

## 累積與年化報酬

以每日投組損益 $\Delta_t$（各配對當日損益加總）建構權益曲線 $E_t = C_0 + \sum_{s\le t}\Delta_s$（$C_0=10{,}000$）。

- **累積報酬（Cum. Return）** $= \prod_{m}(1+r_m) - 1$，其中 $r_m$ 為月頻權益報酬。
- **年化報酬（Ann. Return）** $= (1+\text{Cum})^{12/n_m} - 1$，$n_m$ 為月數（幾何年化）。
- **Final Equity** $= C_0 + \sum_t \Delta_t$（期末權益）。

## 資本效率口徑（文獻慣例）

| 欄位 | 定義 | 文獻依據 |
|---|---|---|
| **RCC**（Return on Committed Capital） | 總損益 ÷ 承諾資本（$C_0$） | Gatev 等人（2006）保守口徑 |
| **REC**（Return on Employed Capital） | 總損益 ÷ 實際動用資本 | Gatev 等人（2006）fully-invested 口徑 |
| **Avg. Utilization** | 日均持倉配對數 ÷ 最大槽位（Top N × 重疊期數） | 資金利用率 |
| **Ann. Ret (Employed)** | 總損益 ÷（日均動用資金 × 年數） | GGR fully-invested 年化 |
| **Excess vs RF** | 承諾資本算術年化 − $r_f\times$利用率 | 閒置現金計無風險利率後的超額 |

> **判讀基準**：Gatev 等人（2006）於 1962–2002 樣本回報 committed 口徑約 11%／年；
> Do 與 Faff（2010）指出 2002 年後扣費淨值趨近於 0。本研究以 committed 口徑為主要基準，
> 並同列 employed 與 rf 超額口徑，忠實反映 post-2002 的報酬衰減。
> `RF_ANNUAL = 0.02`（config），約當 2000–2025 美國 3 個月期國庫券平均。


# 風險調整後指標

以每日報酬 $r_t = \Delta_t / E_{t-1}$ 計算（$E_{t-1}$ 為前一日權益）。

| 指標 | 公式 | 判讀基準 |
|---|---|---|
| **Sharpe** | $\sqrt{252}\cdot \bar r / \sigma_r$ | >1 佳、>0.5 可、<0 劣（年化，$r_f$ 已於權益法內含） |
| **Calmar** | 年化報酬 / \|MDD\| | >0.5 佳；衡量報酬相對最大回撤 |
| **Max Drawdown（MDD）** | $\min_t (E_t - \max_{s\le t}E_s)/\max_{s\le t}E_s$ | 越接近 0 越好（負值） |

**Sharpe（Active Days Only）**：儀表板提供切換，僅計入「有持倉日」的報酬，
排除閒置現金日對波動的稀釋，與 buy-and-hold 的可比性較低但更反映策略活躍期的品質。

> Sharpe 判讀沿用 Sharpe（1994）之慣例；配對交易文獻（如 Do 與 Faff, 2010）
> 於扣費後多落於 0–0.5 區間，故本研究以 **>0.5 為具實務意義**、**顯著 > 基準**為主要目標。


# 交易統計

以「一筆完整交易」（進場至平倉）為單位，`Trade_PnL` 非零者計入。

| 欄位 | 定義 |
|---|---|
| **Win Rate** | 獲利交易數 ÷ 總交易數（≥50% 綠、<50% 紅） |
| **Profit Factor** | 總獲利 ÷ \|總虧損\|（>1 獲利、=1 損益兩平） |
| **Avg Hold (days)** | 平均持倉天數 |
| **Total Trades / Entries / Exits** | 交易筆數、進場次數、正常出場次數 |
| **Forced Closes** | 期末強制平倉次數（交易期結束仍持倉） |
| **Stop Losses** | 觸發停損次數 |
| **Gross Profit / Gross Loss** | 總獲利金額 / 總虧損金額 |

> **注意**：高 Win Rate 未必等於高獲利。若「贏小輸大」（平均獲利 < 平均虧損），
> 即使勝率 > 50%，Profit Factor 仍可能 < 1。本研究多數誠實策略即呈此結構，
> 這是配對交易在扣費後報酬趨零的微觀成因。


# 統計檢定基準（T 檢定）

儀表板的 `T-Stat / p-val / NW T-Stat / NW p-val` 用以檢定**某策略是否顯著優於基準策略**，
而非僅比較單點績效（避免網格選擇偏差）。

## 虛無假設與統計量

對「策略 A」與「基準策略 B」在**共同月份**上計算逐月報酬差 $d_m = r^A_m - r^B_m$，檢定：

$$H_0:\ \mathbb{E}[d_m] = 0 \quad\text{vs}\quad H_1:\ \mathbb{E}[d_m] \ne 0$$

- **一般 t 檢定**：$t = \bar d / (s_d/\sqrt{n})$，$s_d$ 為 $d_m$ 樣本標準差。
- **Newey-West 修正 t 值**：以 Newey 與 West（1987）的 HAC 標準誤修正報酬的
  序列自相關與異質變異（落後階數預設 3），統計量更保守、更穩健。

## 判讀門檻

| p 值 | 標示 | 意義 |
|---|---|---|
| $p < 0.05$ | 綠色粗體 | 5% 水準顯著優於／異於基準 |
| $0.05 \le p < 0.10$ | 橙色 | 10% 水準邊際顯著 |
| $p \ge 0.10$ | 無標示 | 無法拒絕「與基準無差異」 |

> **基準策略的選定**：命題 1（形成法）以 **SSD Rolling**（距離／共整合家族基準）為對照；
> 命題 2（交易法）以**同一組配對的 Z-Score 回歸基準**為對照（DRL／距離策略借用相同形成期
> 配對，構成單變因對照）。因此正的且顯著的 t 值代表「該創新相對基準的淨增量貢獻」。
> 逐月配對檢定與 Newey-West 修正為資產定價實證的標準做法（Newey & West, 1987）。


# 機器學習與深度學習：特徵與訓練資料

本研究兩處使用學習方法：**形成期的非監督聚類**（命題 1）與**交易期的門檻選擇**（命題 2）。
以下完整說明各自的特徵、訓練資料與執行步驟。


## 形成期聚類（非監督式）

**目的**：以資料驅動方式將行為相似的股票分群，縮小配對搜尋空間，取代靜態 GICS 產業分組。

### 特徵

1. **報酬 PCA 因子載荷**（HDBSCAN Cluster / Agglomerative）：對形成期日報酬矩陣（逐股
   標準化 → 等同對相關矩陣）做 PCA，取前 $k$ 個主成分（$k=5\sim15$），每檔股票的載荷
   向量（以 $\sqrt{\text{特徵值}}$ 加權）即其座標。此表徵反映股票在共同風險因子上的暴露
   （Avellaneda & Lee, 2010），是共整合關係的經濟基礎。
2. **公司基本面**（Agglomerative Fundamentals）：對數市值、盈餘殖利率（1/PE），
   以產業中位數插補缺失並 winsorize；Point-in-Time 對齊避免前視。
3. **GICS 產業 one-hot**：加權後併入特徵，軟性引導同產業靠近。

### 演算法與執行步驟

1. 建構特徵矩陣（上述 1–3，各區塊標準化後加權串接）。
2. **HDBSCAN**（Campello 等人, 2013）或 **Agglomerative**（Ward linkage）聚類；
   HDBSCAN 以密度自動決定群數並辨識雜訊點，Agglomerative 以距離門檻（分位數）切群。
3. 群內全配對做共整合／距離篩選（ADF、OU 半衰期、Hurst、零穿越），依 SSD／DTW／
   PCA 融合分數排序取 Top N。

> **無監督、無標籤**：聚類不使用未來報酬，純以形成期資料分群，故無前視風險。


## 交易期門檻選擇（DRL 門檻選擇式 v4）

**目的**：每配對每期自適應選擇進出場門檻，取代固定的 Z-Score 門檻（entry_z=2, exit_z=0）。
設計依 Kim 與 Kim（2019）之門檻選擇框架，動作選單**包含靜態基準**，故策略空間為
Z-Score 基準之超集。

### 輸入特徵（12 維，形成期計算，標準化至約 $[-3,3]$）

以形成期 spread 的 Z 序列與兩檔股票的 log 價格計算：期末 z、$|$期末 z$|$、零穿越頻率、
OU 半衰期（對數）、近期 z 波動 regime、近期 z 趨勢、log 價格相關係數、報酬波動比、
對沖比率、spread 振幅、形成期 $|z|>2$ 佔比、$\max|z|$。這些特徵刻畫配對在交易期
**是否／如何**均值回歸，用以預測各門檻組合的報酬。

### 訓練資料與標籤（全資訊監督回歸）

- **動作選單（9）**：SKIP（不交易）＋ 8 組 $(\text{entry\_z},\text{exit\_z}) \in
  \{1.5,2.0,2.5,3.0\}\times\{0.0,0.5\}$。
- **標籤**：對每一個歷史配對期，**精確反事實回算**全部 9 個動作在該交易期的實際報酬
  （逐一以 Z-Score 狀態機模擬）。因報酬完全可算，此為**全資訊監督回歸**而非部分回饋的
  bandit——無探索問題、樣本效率最高。
- **網路**：MLP（12 → 隱藏層 → 9），輸出各動作的預期報酬，決策時取 argmax。

### Walk-forward 訓練步驟（無前視）

1. 依時間順序走過各滾動期。決策第 $k$ 期時，**僅**以「交易期已於第 $k$ 期開始前結束」
   的歷史配對期樣本訓練（滾動緩衝，重疊期自動排除）。
2. 訓練樣本不足 `thr_min_train_samples`（預設 200）時，**自動退回靜態基準動作 (2.0, 0.0)**，
   確保暖身期行為 $\equiv$ Z-Score 基準。
3. 以選定門檻交由標準 Z-Score 狀態機執行整個交易期，落地交易紀錄。

> **可證偽的比較框架**：因動作選單含基準，DRL 若學得當則 $\ge$ 基準、學不好至多退化為基準；
> 任何顯著正的 t 值即為「自適應門檻相對固定門檻」的淨貢獻。


# 參考文獻

*（APA 第 7 版）*

Avellaneda, M., & Lee, J.-H. (2010). Statistical arbitrage in the US equities market. *Quantitative Finance, 10*(7), 761–782. https://doi.org/10.1080/14697680903124632

Campello, R. J. G. B., Moulavi, D., & Sander, J. (2013). Density-based clustering based on hierarchical density estimates. In *Advances in Knowledge Discovery and Data Mining (PAKDD 2013)* (pp. 160–172). Springer. https://doi.org/10.1007/978-3-642-37456-2_14

Do, B., & Faff, R. (2010). Does simple pairs trading still work? *Financial Analysts Journal, 66*(4), 83–95. https://doi.org/10.2469/faj.v66.n4.1

Do, B., & Faff, R. (2012). Are pairs trading profits robust to trading costs? *Journal of Financial Research, 35*(2), 261–287. https://doi.org/10.1111/j.1475-6803.2012.01317.x

Gatev, E., Goetzmann, W. N., & Rouwenhorst, K. G. (2006). Pairs trading: Performance of a relative-value arbitrage rule. *The Review of Financial Studies, 19*(3), 797–827. https://doi.org/10.1093/rfs/hhj020

Kim, T., & Kim, H. Y. (2019). Optimizing the pairs-trading strategy using deep reinforcement learning with trading and stop-loss boundaries. *Complexity, 2019*, 3582516. https://doi.org/10.1155/2019/3582516

Newey, W. K., & West, K. D. (1987). A simple, positive semi-definite, heteroskedasticity and autocorrelation consistent covariance matrix. *Econometrica, 55*(3), 703–708. https://doi.org/10.2307/1913610

Sharpe, W. F. (1994). The Sharpe ratio. *The Journal of Portfolio Management, 21*(1), 49–58. https://doi.org/10.3905/jpm.1994.409501
